# ch18_economic_analysis_npv — 18.12 NeqSim economic tooling

From chapter `ch18_economic_analysis_npv`, section: **18.12 NeqSim economic tooling**.

This companion notebook reproduces or expands the textbook code block. Run all cells from top to bottom; cells that are intentionally illustrative are marked in the code comments.



In [1]:
from neqsim import jneqsim as ns
import numpy as np

# Build a production profile (Sm³/d gas vs time)
years = np.arange(0, 30)
plateau = 22e6  # Sm³/d
q = np.where(years < 7, plateau, plateau*0.88**(years-7+1))

# Cash flow with the NCS fiscal regime
tax = ns.process.fielddevelopment.economics.NorwegianTaxModel()
engine = ns.process.fielddevelopment.economics.CashFlowEngine(tax)
engine.setGasPrice(0.30)               # USD/Sm³
engine.setFixedOpexPerYear(250e6)      # USD/yr

# CAPEX schedule (4 yrs); years start at 1 (year 0 is reserved)
capex_total = 7_000e6                   # USD
for t, frac in enumerate([0.1, 0.3, 0.4, 0.2]):
    engine.addCapex(float(capex_total*frac), int(t+1))

# Annual production (gas only, oil=ngl=0)
for t, qy in enumerate(q):
    engine.addAnnualProduction(int(t+1), 0.0, float(qy*365.0), 0.0)

discount = 0.08
npv_after_tax = engine.calculateNPV(discount)
print(f"After-tax NPV at {discount*100:.0f}%: {npv_after_tax/1e9:.2f} BUSD")
print(f"Break-even gas price (USD/Sm³): "
      f"{engine.calculateBreakevenGasPrice(discount):.3f}")


After-tax NPV at 8%: -11.67 BUSD
Break-even gas price (USD/Sm³): 1.000
